<a href="https://colab.research.google.com/github/homeroauto-byte/taller_K-Mean-/blob/main/notebooks/Taller_K_Mean_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Taller K _Mean**

**Manuel Gonzalez**  
**Daniel Garcia**  

##**Introduccion a la inteligencia artificial**




<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/homeroauto-byte/taller_K-Mean-/blob/main/notebooks/Taller_K_Mean_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/homeroauto-byte/taller_K-Mean-/blob/main/notebooks/Taller_K_Mean_.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>




## **Vamos con el preambulo de Pyton y las arandelas**

In [ ]:
import sys
import pandas as pd
import urllib.request
import json
import os, re, zipfile, builtins
from datetime import datetime
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
assert sys.version_info >= (3, 10)
from packaging.version import Version
import sklearn
assert Version(sklearn.__version__) >= Version("1.6.1")



# ----------------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL DE ESTILO
# ----------------------------------------------------------------------------
plt.rc('font', size=12)
plt.rc('axes', labelsize=13, titlesize=14)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
plt.rcParams['figure.dpi']  = 100
plt.rcParams['axes.grid']   = True
plt.rcParams['grid.alpha']  = 0.3
sns.set_palette("husl")


# ============================================================================
# SETUP DE EXPORTACIÓN —
# ============================================================================
# --- Carpeta raíz de salidas ---
OUT_DIR = Path("outputs_taller")
FIG_DIR = OUT_DIR / "figures"
TBL_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

# --- Registros globales ---
_TABLES_REGISTRY = {}     # nombre → DataFrame/Series/objeto
_FIG_COUNTER     = {"n": 0}
_TBL_COUNTER     = {"n": 0}

def _slug(s, maxlen=70):
    """Convierte texto en nombre de archivo seguro."""
    s = re.sub(r"[^\w\-]+", "_", str(s)).strip("_")
    return (s[:maxlen] or "sin_titulo")

def register_table(name, obj):
    """Registra manualmente una tabla para exportarla al final."""
    try:
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            _TABLES_REGISTRY[_slug(name)] = obj.copy()
        else:
            _TABLES_REGISTRY[_slug(name)] = obj
    except Exception as e:
        print(f"⚠ register_table('{name}'): {e}")

# ----------------------------------------------------------------------------
# Hook 1: plt.show() → guarda antes de mostrar
# ----------------------------------------------------------------------------
_orig_show = plt.show

def _show_and_save(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _FIG_COUNTER["n"] += 1
        n = _FIG_COUNTER["n"]
        # Nombre preferido: suptitle > título del primer eje > genérico
        title = ""
        if getattr(fig, "_suptitle", None) is not None:
            title = fig._suptitle.get_text()
        elif fig.axes and fig.axes[0].get_title():
            title = fig.axes[0].get_title()
        base = f"fig{n:03d}_{_slug(title)}"
        for ext in ("svg", "pdf", "png"):
            try:
                fig.savefig(FIG_DIR / f"{base}.{ext}",
                            format=ext, bbox_inches="tight",
                            dpi=150, facecolor="white")
            except Exception as e:
                print(f"⚠ No se pudo guardar {base}.{ext}: {e}")
    return _orig_show(*args, **kwargs)

plt.show = _show_and_save

# ----------------------------------------------------------------------------
# Hook 2: display(df) → registra el DataFrame automáticamente
# ----------------------------------------------------------------------------
try:
    from IPython.display import display as _orig_display

    def _display_and_register(*args, **kwargs):
        for a in args:
            if isinstance(a, (pd.DataFrame, pd.Series)):
                _TBL_COUNTER["n"] += 1
                _TABLES_REGISTRY[f"display_{_TBL_COUNTER['n']:03d}"] = a.copy()
        return _orig_display(*args, **kwargs)

    # Sobrescribir en el namespace de builtins y en IPython
    builtins.display = _display_and_register
    try:
        import IPython
        IPython.display.display = _display_and_register
    except Exception:
        pass
except Exception as e:
    print(f"⚠ No se pudo parchear display(): {e}")

print(f"✓ Hooks instalados")
print(f"  Figuras → {FIG_DIR.resolve()}")
print(f"  Tablas  → {TBL_DIR.resolve()}")
print(f"  Tablas manuales: register_table('nombre', df)")


##**Ahora vamos con los datos **
-Base de datos de interconsultas anonimizadas-

In [ ]:
def load_interconsultas_data():
# ----------------------------------------------------------------------------
# 1. Definimos la ruta local donde guardaremos el archivo
# ----------------------------------------------------------------------------
    file_path = Path("datasets/interconsultas.xlsx")

    if not file_path.is_file():
        file_path.parent.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 2. Usamos la API de GitHub para listar los archivos del repositorio
# Esto evita errores por espacios o tildes en el nombre del archivo
# ----------------------------------------------------------------------------
        api_url = "https://api.github.com/repos/homeroauto-byte/taller_K-Mean-/contents/"

        print("Buscando el archivo en el repositorio...")
        try:
            req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response:
                files = json.loads(response.read().decode())

# Buscamos el archivo que termine en .xlsx
            download_url = None
            for file in files:
                if file['name'].endswith('.xlsx'):
                    download_url = file['download_url']
                    print(f"Archivo encontrado: {file['name']}")
                    break

            if not download_url:
                raise FileNotFoundError("No se encontró ningún archivo .xlsx en el repositorio.")

        except Exception as e:
            print(f"Error al conectar con la API de GitHub: {e}")
            print("Intentando con URL directa alternativa...")
# Plan B: URL directa codificada manualmente por si la API falla
            download_url = "https://raw.githubusercontent.com/homeroauto-byte/taller_K-Mean-/main/Taller%20K%20means%20INTERCONSULTAS%20-%20TIEMPOS%20DE%20RESPUESTA.xlsx"
# ----------------------------------------------------------------------------
# 3. Descargamos el archivo
# ----------------------------------------------------------------------------
        print(f"Descargando archivo... (esto puede tardar un poco debido al peso de 20MB)")
        try:
            urllib.request.urlretrieve(download_url, file_path)
            print("Descarga completada exitosamente.")
        except Exception as e:
            print(f"Error al descargar: {e}")
            return None
# ----------------------------------------------------------------------------
# 4. Leemos el archivo Excel
# ----------------------------------------------------------------------------
    print("Cargando datos en Pandas...")
    return pd.read_excel(file_path)

# Ejecutamos la función
df_interconsultas = load_interconsultas_data()

# Mostramos las primeras filas para verificar
if df_interconsultas is not None:
    print("\n¡Datos cargados correctamente!")
    display(df_interconsultas.head())
else:
    print("\nNo se pudieron cargar los datos.")

 Exploración y Diagnóstico Inicial de Datos
### Dataset: Interconsultas Hospitalarias — Tiempos de Respuesta IC
**Objetivo:** Conocer estructura, calidad, tipos de datos y distribución de las variables categóricas antes de   el modelado.

In [ ]:
# ----------------------------------------------------------------------------
# 1. INFORMACIÓN  DEL DATAFRAME
# ----------------------------------------------------------------------------
df_interconsultas.info()

---

## Diagnóstico Inicial

El análisis estructural de `df_interconsultas` revela una base de datos rica en información pero heterogénea, que requiere un proceso de homogeneización y ajuste antes de cualquier análisis o modelado .

### Principales hallazgos

| # | Hallazgo | Implicación |
|---|----------|-------------|
| 1 | **Columnas de fecha en formato `object` (texto)** | No se pueden operar directamente. Requieren conversión a `datetime`. |
| 2 | **`DIAS` y `HORAS` redundantes** | `HORAS` (timedelta) ya contiene el tiempo total; `DIAS` es derivable. Hay que unificar el criterio. |
| 3 | **`HCPRIINTER` es ordinal, no numérico** | Codifica 1=Normal, 2=Prioritario, 3=Urgente. Debe tratarse como categórica ordenada. |  
  




---   
Se realizara un diagnostico mas a fondo   descriptivo y visual para poder  evaluar cambios



In [ ]:
# ----------------------------------------------------------------------------
# 1. INFORMACIÓN ESTRUCTURAL DEL DATAFRAME
# ----------------------------------------------------------------------------
print("=" * 70)
print("1. ESTRUCTURA DEL DATAFRAME")
print("=" * 70)
df_interconsultas.info()
print(f"\nDimensiones: {df_interconsultas.shape[0]:,} filas × {df_interconsultas.shape[1]} columnas")

# Hacemos una copia de trabajo para no modificar el original
df = df_interconsultas.copy()

# ----------------------------------------------------------------------------
# 2. FUNCIÓN DE LIMPIEZA BÁSICA DE TEXTO
#    Normaliza a mayúsculas, quita tildes y espacios duplicados.
#    Se usa aquí solo para agrupar correctamente categorías en el diagnóstico.
# ----------------------------------------------------------------------------
def limpiar(s):
    """Normaliza texto: mayúsculas, sin tildes, sin espacios dobles."""
    if pd.isna(s):
        return np.nan
    return ' '.join(
        str(s).strip().upper()
        .replace('Á', 'A').replace('É', 'E').replace('Í', 'I')
        .replace('Ó', 'O').replace('Ú', 'U').replace('Ñ', 'N')
        .split()
    )

for col in ['ESPECIALIDAD_SOLICITADA', 'AREASOLICITA', 'TIPO_INTERCONSULTA']:
    df[col] = df[col].apply(limpiar)

# Extraer códigos (lo que va antes del guion) para agrupar mejor las categorías
df['AREA_CODIGO'] = df['AREASOLICITA'].str.split(' -').str[0].str.strip()
df['TIPO_CODIGO'] = df['TIPO_INTERCONSULTA'].str.split(' -').str[0].str.strip()

# ----------------------------------------------------------------------------
# 3. CARDINALIDAD DE VARIABLES CATEGÓRICAS (por código, evita duplicados)
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("2. CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print("=" * 70)

print(f"\n>>> ÁREAS ÚNICAS (por código): {df['AREA_CODIGO'].nunique()}")
print(df['AREA_CODIGO'].value_counts().to_string())

print(f"\n>>> TIPOS DE INTERCONSULTA ÚNICOS (por código): {df['TIPO_CODIGO'].nunique()}")
print(df['TIPO_CODIGO'].value_counts().to_string())

print(f"\n>>> ESPECIALIDADES ÚNICAS: {df['ESPECIALIDAD_SOLICITADA'].nunique()}")
print(df['ESPECIALIDAD_SOLICITADA'].value_counts().to_string())

# ----------------------------------------------------------------------------
# 4. DIAGNÓSTICO DE FECHAS Y FORMATO
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("3. DIAGNÓSTICO DE FECHAS")
print("=" * 70)
print("\nMuestra (primeras 3 filas):")
print(df[['FECHA_SOLICITUD', 'FECHA_RESPUESTA']].head(3).to_string())
print(f"\nTipo de dato FECHA_SOLICITUD: {df['FECHA_SOLICITUD'].dtype}")
print(f"Tipo de dato FECHA_RESPUESTA: {df['FECHA_RESPUESTA'].dtype}")

# Detección de valores corruptos tipo #VALUE!, N/A, None, etc.
for col in ['FECHA_SOLICITUD', 'FECHA_RESPUESTA']:
    errores = df[col].astype(str).str.contains('#|N/A|None|nan', case=False, na=False).sum()
    print(f"  {col} con valores corruptos (#, N/A, None): {errores}")

# ----------------------------------------------------------------------------
# 5. DIAGNÓSTICO DE LA VARIABLE TIEMPO (HORAS como timedelta)
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("4. DIAGNÓSTICO DE TIEMPO (columna HORAS)")
print("=" * 70)
horas_num = df['HORAS'].dt.total_seconds() / 3600
print(horas_num.describe().round(3))
print(f"\n  Tiempos negativos (inconsistencias): {(horas_num < 0).sum()}")
print(f"  Tiempos > 30 días (>720h):           {(horas_num > 720).sum()}")

# Comparar DIAS vs HORAS/24 para ver si son redundantes
print("\n### ¿Son DIAS y HORAS redundantes? ###")
print(pd.DataFrame({
    'DIAS':      df['DIAS'],
    'HORAS/24':  horas_num / 24,
    'DIF_abs':   (df['DIAS'] - horas_num / 24).abs()
}).describe().round(4))

# ----------------------------------------------------------------------------
# 6. DIAGNÓSTICO DE LA PRIORIDAD (HCPRIINTER)
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("5. DISTRIBUCIÓN DE PRIORIDAD (HCPRIINTER)")
print("=" * 70)
print(df['HCPRIINTER'].value_counts().sort_index().to_string())
print("\n  Leyenda: 1=Normal, 2=Prioritario, 3=Urgente")

# ----------------------------------------------------------------------------
# 7. CALIDAD DE DATOS: NULOS Y DUPLICADOS
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("6. CALIDAD DE DATOS")
print("=" * 70)
print("\nValores nulos por columna:")
print(df.isna().sum().to_string())
print(f"\nFilas duplicadas exactas: {df.duplicated().sum()}")

# ----------------------------------------------------------------------------
# 8. RESUMEN EJECUTIVO
# ----------------------------------------------------------------------------
resumen = {
    'Registros totales':       f"{len(df):,}",
    'Columnas':                df.shape[1],
    'Valores nulos':           int(df.isna().sum().sum()),
    'Duplicados':              int(df.duplicated().sum()),
    'Áreas únicas':            df['AREA_CODIGO'].nunique(),
    'Tipos IC únicos':         df['TIPO_CODIGO'].nunique(),
    'Especialidades únicas':   df['ESPECIALIDAD_SOLICITADA'].nunique(),
    'Tiempo mediano (h)':      round(horas_num.median(), 2),
    'Tiempo medio (h)':        round(horas_num.mean(), 2),
    'Tiempo máx (h)':          round(horas_num.max(), 2),
}
pd.DataFrame(resumen, index=['Valor']).T

##  — Exploración Visual Inicial (datos crudos)


In [ ]:
# ----------------------------------------------------------------------------
# 1. PREPARACIÓN DE VARIABLES DERIVADAS PARA EL DIAGNÓSTICO
# ----------------------------------------------------------------------------
df = df_interconsultas.copy()

# Fechas
df['FECHA_SOLICITUD'] = pd.to_datetime(df['FECHA_SOLICITUD'], errors='coerce', dayfirst=True)
df['FECHA_RESPUESTA'] = pd.to_datetime(df['FECHA_RESPUESTA'], errors='coerce', dayfirst=True)

# Tiempo de respuesta en horas
df['TIEMPO_HORAS'] = (
    df['FECHA_RESPUESTA'] - df['FECHA_SOLICITUD']
).dt.total_seconds() / 3600

# HORAS (columna original timedelta) → horas numéricas
df['HORAS_NUM'] = df['HORAS'].dt.total_seconds() / 3600

# Códigos de categorías para gráficos de barras
df['AREA_CODIGO'] = df['AREASOLICITA'].str.split(' -').str[0].str.strip()
df['TIPO_CODIGO'] = df['TIPO_INTERCONSULTA'].str.split(' -').str[0].str.strip()

# Prioridad legible
df['PRIORIDAD_LABEL'] = df['HCPRIINTER'].map({1: 'Normal', 2: 'Prioritario', 3: 'Urgente'})

# ----------------------------------------------------------------------------
# 2. HISTOGRAMAS DE VARIABLES NUMÉRICAS (escala lineal)
# ----------------------------------------------------------------------------
num_df = df[['TIEMPO_HORAS', 'HORAS_NUM', 'DIAS', 'HCPRIINTER']].copy()
num_df.columns = ['Tiempo IC (h)', 'HORAS (timedelta → h)', 'DIAS (entero)', 'Prioridad (1-3)']

num_df.hist(bins=50, figsize=(14, 8), color='steelblue', edgecolor='black')
plt.suptitle('Distribución de variables numéricas — DATOS CRUDOS',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 3. HISTOGRAMAS EN ESCALA LOGARÍTMICA (para ver mejor la cola)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tiempo crudo
axes[0].hist(df['TIEMPO_HORAS'].dropna(), bins=80,
             color='coral', edgecolor='black')
axes[0].set_xlabel('Tiempo de respuesta (horas)')
axes[0].set_ylabel('Frecuencia (log)')
axes[0].set_title('Escala lineal — cola a la derecha')
axes[0].set_yscale('log')

# Log del tiempo
log_t = np.log1p(df['TIEMPO_HORAS'].dropna())
axes[1].hist(log_t, bins=80, color='seagreen', edgecolor='black')
axes[1].set_xlabel('log(1 + Tiempo horas)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Escala logarítmica — casi normal')

plt.suptitle('Efecto de la transformación logarítmica sobre el tiempo',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 4. BOXPLOTS — Detección visual de outliers y comparación por prioridad
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 4.1 Boxplot global del tiempo
axes[0].boxplot(df['TIEMPO_HORAS'].dropna(),
                vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_ylabel('Tiempo de respuesta (horas)')
axes[0].set_title('Boxplot global — outliers > 100 h visibles')
axes[0].set_yscale('log')

# 4.2 Boxplot por prioridad
data_by_priority = [
    df[df['HCPRIINTER'] == i]['TIEMPO_HORAS'].dropna()
    for i in [1, 2, 3]
]
bp = axes[1].boxplot(data_by_priority,
                     labels=['Normal', 'Prioritario', 'Urgente'],
                     patch_artist=True)
for patch, color in zip(bp['boxes'], ['#8dd3c7', '#ffffb3', '#fb8072']):
    patch.set_facecolor(color)
axes[1].set_ylabel('Tiempo de respuesta (horas)')
axes[1].set_title('Tiempo por prioridad de la IC')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 5. MAPA DE CALOR — Correlaciones entre variables numéricas
# ----------------------------------------------------------------------------
corr_df = df[['TIEMPO_HORAS', 'DIAS', 'HCPRIINTER']].copy()
corr_df['HORA_DIA']      = df['FECHA_SOLICITUD'].dt.hour
corr_df['DIA_SEMANA']    = df['FECHA_SOLICITUD'].dt.dayofweek
corr_df['ES_FIN_SEMANA'] = (corr_df['DIA_SEMANA'] >= 5).astype(int)

corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'label': 'Coeficiente de correlación'})
plt.title('Mapa de calor — Correlaciones (datos crudos)')
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 6. DISTRIBUCIÓN DE CATEGÓRICAS PRINCIPALES (Top 15 + Prioridad)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 6.1 Top 15 áreas
top_areas = df['AREA_CODIGO'].value_counts().head(15)
top_areas.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Nº de interconsultas')
axes[0].set_title('Top 15 áreas solicitantes')
axes[0].invert_yaxis()

# 6.2 Top 15 tipos de IC
top_tipos = df['TIPO_CODIGO'].value_counts().head(15)
top_tipos.plot(kind='barh', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_xlabel('Nº de interconsultas')
axes[1].set_title('Top 15 tipos de interconsulta')
axes[1].invert_yaxis()

# 6.3 Prioridad
prioridad_counts = df['PRIORIDAD_LABEL'].value_counts().reindex(['Normal', 'Prioritario', 'Urgente'])
prioridad_counts.plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='black')
axes[2].set_ylabel('Nº de interconsultas')
axes[2].set_title('Distribución por prioridad')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Distribución de variables categóricas — DATOS CRUDOS',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 7. RESUMEN DE HALLAZGOS CLAVE
# ----------------------------------------------------------------------------
print("=" * 70)
print("🔍 HALLAZGOS CLAVE DE LA EXPLORACIÓN VISUAL")
print("=" * 70)
print(f"  • Tiempo medio:    {df['TIEMPO_HORAS'].mean():.2f} h")
print(f"  • Tiempo mediano:  {df['TIEMPO_HORAS'].median():.2f} h")
print(f"  • Tiempo máx:      {df['TIEMPO_HORAS'].max():.2f} h ({df['TIEMPO_HORAS'].max()/24:.1f} días)")
print(f"  • Asimetría:       {df['TIEMPO_HORAS'].skew():.2f}  →  justifica log-transform")
print(f"  • Outliers > 100h: {(df['TIEMPO_HORAS'] > 100).sum():,} registros")
print(f"  • Área dominante:  {df['AREA_CODIGO'].mode()[0]} "
      f"({df['AREA_CODIGO'].value_counts().iloc[0]:,} casos)")
print(f"  • IC más frecuente:{df['TIPO_CODIGO'].mode()[0]} "
      f"({df['TIPO_CODIGO'].value_counts().iloc[0]:,} casos)")
print("=" * 70)
for dias in [1, 2, 3, 4, 5, 7, 10]:
    n_out = (df_original['TIEMPO_HORAS'] > dias*24).sum()
    pct   = 100 * n_out / len(df_original)
    print(f"Umbral > {dias:>2} días:  elimina {n_out:>7,} IC ({pct:>5.2f}%)")

## Limpieza Profunda y Homologación Final

### Objetivo
Dejar el dataset de interconsultas listo garantizando:
- Consistencia de tipos de datos
- Calidad y completitud de la información
- Homogeneidad de categorías

| # | Decisión | Justificación |
|---|----------|---------------|
| 1 | **Convertir fechas a `datetime` con `dayfirst=True`** | Formato original `dd/mm/yyyy HH:MM`; sin esto no se pueden calcular diferencias temporales. |
| 2 | **Calcular `TIEMPO_HORAS` desde las fechas, no desde `HORAS`** | Variable independiente y validable. La diferencia media contra la columna `HORAS` es prácticamente nula (consistente entre ambas fuentes). |
| 3 | **Separar IC > 2 días (48 h) en `df_slow`** | La exploración previa mostró cola larga (tiempos de hasta varias semanas). Estas IC son casos atípicos y se conservan aparte para análisis complementario. El porcentaje exacto depende de la corrida. |
| 4 | **Filtrar `df` a [0, 2 días]** | Mejora la interpretabilidad de los clusters operativos. Se pierde ≈3% de los datos (todos ellos casos atípicos de muy larga duración). |
| 5 | **Eliminar duplicados exactos** | Robustez del dataset. |
| 6 | **Normalizar texto: mayúsculas, sin tildes, espacios simples** | Corrige duplicados ocultos tipo `ESGE -  ESTANCIA GENERAL` (doble espacio) vs `ESGE - ESTANCIA GENERAL`. |
| 7 | **Separar código y nombre en `AREA` y `TIPO`** | El formato es `CÓDIGO - Descripción`. Separarlos permite trabajar con el código (compacto) y conservar el nombre legible. |
| 8 | **Extraer `ESPECIALIDAD` sin prefijo de código** | Se limpia el prefijo tipo `MEDI - ` para tener la especialidad pura. |
| 9 | **Convertir `HCPRIINTER` a categórica ordinal legible** | 1 = Normal, 2 = Prioritario, 3 = Urgente. Se crean dummies `ES_URGENTE` y `ES_PRIORITARIO` para features. |
| 10 | **Crear variables temporales** (`ANIO`, `MES`, `DIA_SEMANA`, `HORA_DIA`, `ES_FIN_SEMANA`, `ES_NOCTURNO`) | Capturan patrones de solicitud (nocturno, fin de semana) útiles para K-Means. |
| 11 | **Aplicar `LOG_TIEMPO = log1p(TIEMPO_HORAS)`** | Asimetría severa (skew claramente > 5). El logaritmo normaliza y estabiliza la varianza. |
| 12 | **Agrupar categorías raras (< 500 casos) en `OTRAS` / `OTROS`** | Hay del orden de ~30 áreas y ~70 tipos, pero solo una decena concentra la mayor parte de los casos. El umbral es configurable en código (`UMBRAL_AREA`, `UMBRAL_TIPO`). Reduce dimensionalidad sin perder información. |





In [ ]:
# ============================================================================
# 1. — LIMPIEZA Y HOMOLOGACIÓN
# ============================================================================
df_original = df_interconsultas.copy()
df = df_interconsultas.copy()

print("=" * 70)
print("🧹 SECCIÓN 1 — LIMPIEZA Y HOMOLOGACIÓN")
print("=" * 70)
print(f"Filas iniciales: {len(df):,}\n")

# --- 1. Fechas ---
df['FECHA_SOLICITUD'] = pd.to_datetime(df['FECHA_SOLICITUD'], errors='coerce', dayfirst=True)
df['FECHA_RESPUESTA']  = pd.to_datetime(df['FECHA_RESPUESTA'],  errors='coerce', dayfirst=True)
n0 = len(df)
df = df.dropna(subset=['FECHA_SOLICITUD', 'FECHA_RESPUESTA'])
print(f"  ✗ Eliminadas por fechas inválidas:              {n0 - len(df):>8,}")

# --- 2. Tiempo en horas ---
df['TIEMPO_HORAS'] = (
    df['FECHA_RESPUESTA'] - df['FECHA_SOLICITUD']
).dt.total_seconds() / 3600
diff_validacion = (df['TIEMPO_HORAS'] - df['HORAS'].dt.total_seconds()/3600).abs()
print(f"  ✓ Diferencia media vs columna HORAS:            {diff_validacion.mean():.4f} h")

# --- 3. Filtros de calidad ---
# 3a. Separar outliers lentos (> 2 días) para análisis complementario
df_slow = df[df['TIEMPO_HORAS'] > 2 * 24].copy()
df      = df[(df['TIEMPO_HORAS'] >= 0) & (df['TIEMPO_HORAS'] <= 2 * 24)].copy()

n_total_antes_filtro = len(df) + len(df_slow)
print(f"  ⚠ Separadas como IC lentas (> 2 d):             {len(df_slow):>8,}  "
      f"({100*len(df_slow)/n_total_antes_filtro:.2f}%)")
print(f"  ✓ Conservadas para K-Means [0, 2 d]:            {len(df):>8,}")

# 3b. Duplicados
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"  ✗ Eliminadas por duplicados exactos:            {n0 - len(df):>8,}")

# --- 4. Homologación de texto ---
def limpiar(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().upper()
    for a, b in zip('ÁÉÍÓÚÑ', 'AEIOUN'):
        s = s.replace(a, b)
    return ' '.join(s.split())

for col in ['ESPECIALIDAD_SOLICITADA', 'AREASOLICITA', 'TIPO_INTERCONSULTA']:
    df[col] = df[col].apply(limpiar)
    df_slow[col] = df_slow[col].apply(limpiar)

df['AREA_CODIGO'] = df['AREASOLICITA'].str.split(' -').str[0].str.strip()
df['AREA_NOMBRE'] = df['AREASOLICITA'].str.split(' -').str[-1].str.strip()
df['TIPO_CODIGO'] = df['TIPO_INTERCONSULTA'].str.split(' -').str[0].str.strip()
df['TIPO_NOMBRE'] = df['TIPO_INTERCONSULTA'].str.split(' -').str[-1].str.strip()
df['ESPECIALIDAD'] = (df['ESPECIALIDAD_SOLICITADA']
                     .str.replace(r'^[A-Z]+ - ?', '', regex=True)
                     .str.strip())

df_slow['AREA_CODIGO'] = df_slow['AREASOLICITA'].str.split(' -').str[0].str.strip()
df_slow['TIPO_CODIGO'] = df_slow['TIPO_INTERCONSULTA'].str.split(' -').str[0].str.strip()
df_slow['ESPECIALIDAD'] = (df_slow['ESPECIALIDAD_SOLICITADA']
                          .str.replace(r'^[A-Z]+ - ?', '', regex=True)
                          .str.strip())

# --- 5. Prioridad ---
df['PRIORIDAD']      = df['HCPRIINTER'].map({1: 'NORMAL', 2: 'PRIORITARIO', 3: 'URGENTE'})
df['ES_URGENTE']     = (df['HCPRIINTER'] == 3).astype(int)
df['ES_PRIORITARIO'] = (df['HCPRIINTER'] == 2).astype(int)

df_slow['PRIORIDAD'] = df_slow['HCPRIINTER'].map({1: 'NORMAL', 2: 'PRIORITARIO', 3: 'URGENTE'})

# --- 6. Variables temporales ---
df['ANIO']          = df['FECHA_SOLICITUD'].dt.year
df['MES']           = df['FECHA_SOLICITUD'].dt.month
df['DIA_SEMANA']    = df['FECHA_SOLICITUD'].dt.dayofweek
df['HORA_DIA']      = df['FECHA_SOLICITUD'].dt.hour
df['ES_FIN_SEMANA'] = (df['DIA_SEMANA'] >= 5).astype(int)
df['ES_NOCTURNO']   = ((df['HORA_DIA'] >= 19) | (df['HORA_DIA'] <= 6)).astype(int)

df_slow['ANIO']          = df_slow['FECHA_SOLICITUD'].dt.year
df_slow['MES']           = df_slow['FECHA_SOLICITUD'].dt.month
df_slow['DIA_SEMANA']    = df_slow['FECHA_SOLICITUD'].dt.dayofweek
df_slow['HORA_DIA']      = df_slow['FECHA_SOLICITUD'].dt.hour
df_slow['ES_FIN_SEMANA'] = (df_slow['DIA_SEMANA'] >= 5).astype(int)
df_slow['ES_NOCTURNO']   = ((df_slow['HORA_DIA'] >= 19) | (df_slow['HORA_DIA'] <= 6)).astype(int)

# --- 7. Log-transform ---
df['LOG_TIEMPO'] = np.log1p(df['TIEMPO_HORAS'])

# --- 8. Agrupación de categorías raras ---
UMBRAL_AREA = 500
UMBRAL_TIPO = 500
areas_ok = df['AREA_CODIGO'].value_counts()
df['AREA_GRUPO'] = df['AREA_CODIGO'].where(
    df['AREA_CODIGO'].map(areas_ok) >= UMBRAL_AREA, 'OTRAS'
)
tipos_ok = df['TIPO_CODIGO'].value_counts()
df['TIPO_GRUPO'] = df['TIPO_CODIGO'].where(
    df['TIPO_CODIGO'].map(tipos_ok) >= UMBRAL_TIPO, 'OTROS'
)

# Aplicar misma agrupación a df_slow (usando los grupos aprendidos de df)
df_slow['AREA_GRUPO'] = df_slow['AREA_CODIGO'].where(
    df_slow['AREA_CODIGO'].isin(df['AREA_GRUPO'].unique()), 'OTRAS'
)
df_slow['TIPO_GRUPO'] = df_slow['TIPO_CODIGO'].where(
    df_slow['TIPO_CODIGO'].isin(df['TIPO_GRUPO'].unique()), 'OTROS'
)

# ============================================================================
# 2. — RESUMEN Y VERIFICACIÓN
# ============================================================================
print("\n" + "=" * 70)
print("✅ SECCIÓN 2 — RESUMEN Y VERIFICACIÓN DEL DATASET LIMPIO")
print("=" * 70)
print(f"  Filas finales (df):          {len(df):>8,}")
print(f"  Filas IC lentas (df_slow):   {len(df_slow):>8,}")
print(f"  Columnas finales:            {df.shape[1]:>8}")
print(f"  Reducción total:             {100*(1 - len(df)/len(df_original)):>7.2f}%")
print(f"  Áreas únicas (grupo):        {df['AREA_GRUPO'].nunique():>8} (de {df['AREA_CODIGO'].nunique()})")
print(f"  Tipos IC únicos (grupo):     {df['TIPO_GRUPO'].nunique():>8} (de {df['TIPO_CODIGO'].nunique()})")
print(f"  Especialidades únicas:       {df['ESPECIALIDAD'].nunique():>8}")
print(f"  Tiempo mediano (df):         {df['TIEMPO_HORAS'].median():>7.2f} h")
print(f"  Tiempo medio (df):           {df['TIEMPO_HORAS'].mean():>7.2f} h")
print(f"  Tiempo máximo (df):          {df['TIEMPO_HORAS'].max():>7.2f} h")
print(f"  Skew del tiempo (df):        {df['TIEMPO_HORAS'].skew():>8.2f}")
print(f"  Nulos totales:               {df.isna().sum().sum():>8,}")
print(f"  Duplicados exactos:          {df.duplicated().sum():>8,}")

print("\n📊 Distribución por Área (grupo):")
print(df['AREA_GRUPO'].value_counts().to_string())

print("\n📊 Distribución por Prioridad:")
print(df['PRIORIDAD'].value_counts().to_string())

# ============================================================================
# 3. — VISUALIZACIÓN COMPARATIVA ANTES vs DESPUÉS
# ============================================================================
print("\n" + "=" * 70)
print("📊 SECCIÓN 3 — VISUALIZACIÓN COMPARATIVA")
print("=" * 70)

df_original['TIEMPO_HORAS'] = (
    pd.to_datetime(df_original['FECHA_RESPUESTA'], dayfirst=True, errors='coerce') -
    pd.to_datetime(df_original['FECHA_SOLICITUD'], dayfirst=True, errors='coerce')
).dt.total_seconds() / 3600

# -------- Figura 1: comparativa ANTES vs DESPUÉS --------
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(df_original['TIEMPO_HORAS'].dropna(), bins=80,
                color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('ANTES — tiempo crudo (con outliers)')
axes[0, 0].set_xlabel('Tiempo (h)'); axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].hist(df['TIEMPO_HORAS'], bins=80,
                color='mediumseagreen', edgecolor='black', alpha=0.8)
axes[0, 1].set_title('DESPUÉS — tiempo limpio [0, 2 d]')
axes[0, 1].set_xlabel('Tiempo (h)'); axes[0, 1].set_ylabel('Frecuencia')

axes[1, 0].boxplot(df['TIEMPO_HORAS'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightblue'),
                   medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_title('Boxplot del tiempo limpio [0, 2 d]')
axes[1, 0].set_ylabel('Tiempo (h)'); axes[1, 0].set_yscale('log')

prioridad_counts = df['PRIORIDAD'].value_counts().reindex(['NORMAL', 'PRIORITARIO', 'URGENTE'])
colors = ['#8dd3c7', '#ffffb3', '#fb8072']
axes[1, 1].bar(prioridad_counts.index, prioridad_counts.values,
               color=colors, edgecolor='black')
axes[1, 1].set_title('Distribución por prioridad (limpio)')
axes[1, 1].set_ylabel('Nº de IC')

plt.suptitle('📊 Comparativa ANTES vs DESPUÉS de la limpieza', fontsize=15)
plt.tight_layout(); plt.show()

# -------- Figura 2: series temporales --------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby(['ANIO', 'MES']).size().plot(ax=axes[0], color='steelblue', marker='o')
axes[0].set_title('Nº de interconsultas por mes')
axes[0].set_xlabel('Año-Mes'); axes[0].set_ylabel('Nº de IC')

df['HORA_DIA'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Distribución horaria de solicitudes')
axes[1].set_xlabel('Hora del día'); axes[1].set_ylabel('Nº de IC')

plt.tight_layout(); plt.show()

# -------- Figura 3: tiempo mediano por área --------
top_areas = df['AREA_GRUPO'].value_counts().head(12).index
mediana_por_area = (df[df['AREA_GRUPO'].isin(top_areas)]
                    .groupby('AREA_GRUPO')['TIEMPO_HORAS']
                    .median()
                    .sort_values(ascending=True))

plt.figure(figsize=(10, 6))
mediana_por_area.plot(kind='barh', color='mediumpurple', edgecolor='black')
plt.xlabel('Tiempo mediano de respuesta (h)')
plt.ylabel('Área')
plt.title('Tiempo mediano de respuesta por área (Top 12)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

# ============================================================================
# 4. — ANÁLISIS COMPLEMENTARIO DE IC LENTAS (> 2 días)
# ============================================================================
print("\n" + "=" * 70)
print(f"🔍 SECCIÓN 4 — IC LENTAS (> 2 días): {len(df_slow):,} casos")
print("=" * 70)

if len(df_slow) > 0:
    print(f"  Tiempo mediano:  {df_slow['TIEMPO_HORAS'].median():.1f} h "
          f"({df_slow['TIEMPO_HORAS'].median()/24:.1f} días)")
    print(f"  Tiempo máximo:   {df_slow['TIEMPO_HORAS'].max():.1f} h "
          f"({df_slow['TIEMPO_HORAS'].max()/24:.1f} días)")

    print("\n  📍 Top 5 áreas con IC lentas:")
    print(df_slow['AREA_GRUPO'].value_counts().head(5).to_string())

    print("\n  📍 Top 5 tipos de IC lentas:")
    print(df_slow['TIPO_GRUPO'].value_counts().head(5).to_string())

    print("\n  📍 Distribución por prioridad:")
    print(df_slow['PRIORIDAD'].value_counts().to_string())

    # Comparativa visual
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    top_a = df_slow['AREA_GRUPO'].value_counts().head(8)
    top_a.plot(kind='barh', ax=axes[0], color='firebrick', edgecolor='black')
    axes[0].set_title(f'Áreas con IC lentas (> 2 d) — n={len(df_slow):,}')
    axes[0].set_xlabel('Nº de casos')
    axes[0].invert_yaxis()

    df_slow['PRIORIDAD'].value_counts().reindex(
        ['NORMAL', 'PRIORITARIO', 'URGENTE']
    ).plot(kind='bar', ax=axes[1], color=['#8dd3c7', '#ffffb3', '#fb8072'],
           edgecolor='black')
    axes[1].set_title('Prioridad de las IC lentas')
    axes[1].set_ylabel('Nº de casos')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout(); plt.show()
else:
    print("  No hay IC con duración > 2 días.")

print("\n" + "=" * 70)
print("✅  — Datasets listos:")
print(f"   • df      → {len(df):,} IC  (0 - 2 días)")
print(f"   • df_slow → {len(df_slow):,} IC lentas para análisis complementario")
print("=" * 70)

# **K-Means** — Clustering de Especialidades  

## ¿Qué vamos a hacer?
Aplicar **K-Means**   para agrupar las especialidades médicas según su perfil operativo de respuesta a interconsultas, es decir: qué tan rápido responden, con qué variabilidad, y bajo qué condiciones (urgencia, nocturnidad, fin de semana).

> En lugar de clusterizar las 455k interconsultas individuales, aquí agregamos primero por especialidad para obtener una fila por cada una (~50-80 filas).

## Metodología

1. **Agregación** — Construir un perfil numérico por especialidad con 9 features.
2. **Escalado** — `StandardScaler` .
3. **Selección de K**:
   - **Método del codo** → `kmeans.inertia_`
   - **Silhouette score** → `silhouette_score`
4. **K-Means final** con el K elegido.
5. **Visualizaciones**:
   - **PCA 2D** para ver los clusters proyectados
   - **Silhouette diagram**
6. **Perfilado de clusters** — describir cada grupo con sus especialidades.

##  Features del perfil de cada especialidad

| Feature | Interpretación |
|---|---|
| `tiempo_medio`, `tiempo_mediano` | Velocidad promedio de respuesta |
| `p25`, `p75`, `p95` | Distribución interna (dispersión) |
| `tiempo_std` | Variabilidad (previsibilidad del servicio) |
| `pct_urgentes` | % de IC clasificadas como urgentes |
| `pct_fin_semana` | % de IC solicitadas en fin de semana |
| `pct_nocturno` | % de IC solicitadas en horario nocturno |



##**Paquetes de codigo**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter


In [ ]:
# ============================================================================
# 1. AGREGACIÓN POR ESPECIALIDAD — 1 fila = 1 especialidad
# ============================================================================
perfil = df.groupby('ESPECIALIDAD').agg(
    n_total          = ('TIEMPO_HORAS', 'count'),
    tiempo_medio     = ('TIEMPO_HORAS', 'mean'),
    tiempo_mediano   = ('TIEMPO_HORAS', 'median'),
    p25              = ('TIEMPO_HORAS', lambda x: x.quantile(0.25)),
    p75              = ('TIEMPO_HORAS', lambda x: x.quantile(0.75)),
    p95              = ('TIEMPO_HORAS', lambda x: x.quantile(0.95)),
    tiempo_std       = ('TIEMPO_HORAS', 'std'),
    pct_urgentes     = ('ES_URGENTE',     'mean'),
    pct_prioritarios = ('ES_PRIORITARIO', 'mean'),
    pct_fin_semana   = ('ES_FIN_SEMANA',  'mean'),
    pct_nocturno     = ('ES_NOCTURNO',    'mean'),
).reset_index()

# Filtramos especialidades con al menos 300 casos (perfiles estables)
UMBRAL_N = 300
perfil = perfil[perfil['n_total'] >= UMBRAL_N].reset_index(drop=True)

print("=" * 70)
print(f"— Clustering por especialidad")
print("=" * 70)
print(f"  Especialidades a clusterizar: {len(perfil)}")
print(f"  Filtro aplicado: n_total >= {UMBRAL_N} casos")
print(f"  Total ICs cubiertas: {perfil['n_total'].sum():,}")
print("=" * 70)

# ============================================================================
# 2. FEATURES + ESCALADO
# ============================================================================
features = ['tiempo_medio', 'tiempo_mediano', 'p25', 'p75', 'p95',
            'tiempo_std', 'pct_urgentes', 'pct_fin_semana', 'pct_nocturno']

X = perfil[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================================
# 3. SELECCIÓN DE K — CODO + SILUETA (Fig. 8-7 y 8-8 del libro)
# ============================================================================
inercias  = []
siluetas  = []
K_range   = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(K_range, inercias, "bo-")
ax[0].set_xlabel("$k$")
ax[0].set_ylabel("Inercia")
ax[0].set_title("Método del codo")
ax[0].grid(True)

ax[1].plot(K_range, siluetas, "ro-")
ax[1].set_xlabel("$k$")
ax[1].set_ylabel("Silhouette score")
ax[1].set_title("Silueta por número de clusters")
ax[1].grid(True)

plt.tight_layout()
plt.show()

# Reporte numérico de la silueta
best_k = list(K_range)[int(np.argmax(siluetas))]
print("\n📊 Silueta por k:")
for k, s in zip(K_range, siluetas):
    marca = " ← mejor" if k == best_k else ""
    print(f"   k={k:>2}  silueta={s:.4f}{marca}")
print(f"\n>>> Mejor k según silueta: {best_k}")

# ============================================================================
# 4. K-MEANS FINAL
# ============================================================================
# ⚠️ Ajusta K_FINAL según el gráfico anterior (sugerencia: el mejor de silueta
#    o donde el codo se aplane). Por defecto dejamos el mejor automático.
K_FINAL = best_k

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
perfil['CLUSTER'] = km_final.fit_predict(X_scaled)

print("\n" + "=" * 70)
print(f"✅ K-Means final con k={K_FINAL}")
print(f"   Inercia: {km_final.inertia_:.2f}")
print(f"   Silueta: {silhouette_score(X_scaled, perfil['CLUSTER']):.4f}")
print("=" * 70)

# ============================================================================
# 5. PERFIL DE CADA CLUSTER — tabla numérica
# ============================================================================
resumen = perfil.groupby('CLUSTER')[features].mean().round(2)
resumen['n_especialidades'] = perfil.groupby('CLUSTER').size()
resumen['n_casos_total']    = perfil.groupby('CLUSTER')['n_total'].sum()
resumen['pct_del_total']    = (100 * resumen['n_casos_total'] / perfil['n_total'].sum()).round(2)

print("\n📋 PERFIL NUMÉRICO DE CADA CLUSTER:")
print(resumen.T.to_string())

# ============================================================================
# 6. ESPECIALIDADES ASIGNADAS A CADA CLUSTER
# ============================================================================
print("\n" + "=" * 70)
print("🏥 ESPECIALIDADES POR CLUSTER")
print("=" * 70)

for c in sorted(perfil['CLUSTER'].unique()):
    esp = perfil[perfil['CLUSTER'] == c].sort_values('n_total', ascending=False)
    print(f"\n--- CLUSTER {c} ({len(esp)} especialidades · {esp['n_total'].sum():,} ICs) ---")
    for _, r in esp.head(15).iterrows():
        print(f"   {r['ESPECIALIDAD'][:52]:<54}"
              f"n={int(r['n_total']):>6,}  "
              f"med={r['tiempo_mediano']:>5.1f}h  "
              f"urg={r['pct_urgentes']*100:>4.1f}%")

# ============================================================================
# 7. VISUALIZACIÓN CON PCA 2D (proyección de clusters)
# ============================================================================
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
perfil['PC1'], perfil['PC2'] = X_pca[:, 0], X_pca[:, 1]

plt.figure(figsize=(11, 7))
scatter = plt.scatter(perfil['PC1'], perfil['PC2'],
                      c=perfil['CLUSTER'], cmap='tab10',
                      s=np.sqrt(perfil['n_total']) * 3,
                      alpha=0.75, edgecolor='black')
plt.colorbar(scatter, label='Cluster')

for _, row in perfil.iterrows():
    plt.annotate(row['ESPECIALIDAD'][:18], (row['PC1'], row['PC2']),
                 fontsize=7, alpha=0.85)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title(f"Clusters de especialidades (k={K_FINAL}) — tamaño = nº ICs")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# 8. SILHOUETTE DIAGRAM (Fig. 8-9 del libro)
# ============================================================================
plt.figure(figsize=(11, 7))

y_pred = perfil['CLUSTER'].values
silhouette_vals = silhouette_samples(X_scaled, y_pred)

padding = len(X_scaled) // 30
pos = padding
ticks = []
for i in range(K_FINAL):
    coeffs = silhouette_vals[y_pred == i]
    coeffs.sort()
    color = plt.cm.Spectral(i / K_FINAL)
    plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                      facecolor=color, edgecolor=color, alpha=0.7)
    ticks.append(pos + len(coeffs) // 2)
    pos += len(coeffs) + padding

plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
plt.gca().yaxis.set_major_formatter(FixedFormatter(range(K_FINAL)))
plt.ylabel("Cluster")
plt.xlabel("Coeficiente de silueta")
plt.axvline(x=silhouette_score(X_scaled, y_pred), color="red", linestyle="--",
            label=f"Media = {silhouette_score(X_scaled, y_pred):.3f}")
plt.legend()
plt.title(f"Silhouette diagram (k={K_FINAL})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# 9. EXPORTAR PERFIL PARA EL INFORME
# ============================================================================
perfil_export = perfil[['ESPECIALIDAD', 'n_total', 'CLUSTER'] + features].copy()
perfil_export = perfil_export.sort_values(['CLUSTER', 'n_total'], ascending=[True, False])

# Descomenta si quieres guardar a Excel:
# perfil_export.to_excel("clusters_especialidades.xlsx", index=False)
# print("💾 Guardado: clusters_especialidades.xlsx")


print(f"   • {len(perfil)} especialidades clusterizadas en {K_FINAL} grupos")
print(f"   • Cobertura: {perfil['n_total'].sum():,} ICs ({100*perfil['n_total'].sum()/len(df):.1f}% del dataset)")
print("=" * 70)

# MiniBatchKMeans por interconsulta

Tras la aproximación anterior (que clusterizó las especialidades con KMeans agregado, con `k` elegido automáticamente por silueta), damos el siguiente paso: aplicar **MiniBatchKMeans sobre las interconsultas individuales** —sin agregar— para descubrir perfiles operativos de **cada solicitud** en lugar de cada servicio.

## ¿Por qué MiniBatchKMeans?
El Cap. 8 lo recomienda cuando `n` es grande: entrena con mini-lotes, es **mucho más rápido** que K-Means clásico sin sacrificar calidad (lo demuestra con MNIST). Con cientos de miles de filas, es la herramienta natural.

## Features
- **5 numéricas**: `LOG_TIEMPO`, `HORA_DIA`, `ES_FIN_SEMANA`, `ES_NOCTURNO`, `HCPRIINTER`.
- **N one-hot**: `AREA_GRUPO` (se excluye `PRIORIDAD` porque duplica `HCPRIINTER`). El número exacto depende de cuántas áreas queden tras la agrupación de categorías raras.

## Pipeline
Escalado → barrido `k=2..10` (codo + silueta sobre muestra de 20k) → modelo final → perfilado + PCA.

## Diferencia con la aproximación anterior
| | Aproximación anterior | MiniBatchKMeans |
|---|---|---|
| Unidad | Especialidad | IC |
| Pregunta | ¿Qué servicios se parecen? | ¿Qué **tipos de IC** existen? |

Las dos son complementarias.

##**Paquetes de codigo**

In [ ]:
from sklearn.cluster import MiniBatchKMeans

In [ ]:
# ----------------------------------------------------------------------------
# 0.   — Semilla,
# ----------------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")

# ----------------------------------------------------------------------------
# 1. MATRIZ DE FEATURES  (numéricas + one-hot AREA_GRUPO, sin PRIORIDAD)
# ----------------------------------------------------------------------------
num_cols = ["LOG_TIEMPO", "HORA_DIA", "ES_FIN_SEMANA", "ES_NOCTURNO", "HCPRIINTER"]
cat_cols = ["AREA_GRUPO"]          # PRIORIDAD excluida: duplica HCPRIINTER

X_num = df[num_cols].astype(np.float32).reset_index(drop=True)
X_cat = pd.get_dummies(df[cat_cols].astype(str),
                       prefix=cat_cols, drop_first=False, dtype=np.float32)

X = pd.concat([X_num, X_cat], axis=1)
feature_names = X.columns.tolist()

print(f"Shape de X           : {X.shape}")
print(f"Features numéricas   : {num_cols}")
print(f"One-hot AREA_GRUPO   : {X_cat.shape[1]} columnas "
      f"({df['AREA_GRUPO'].nunique()} áreas)")

# ----------------------------------------------------------------------------
# 2. ESCALADO (StandardScaler — obligatorio en K-Means, Cap. 8)
# ----------------------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)
print(f"X_scaled             : {X_scaled.shape}  |  "
      f"media≈{X_scaled.mean():.3f}  std≈{X_scaled.std():.3f}")

# ----------------------------------------------------------------------------
# 3. BARRIDO DE k — codo (inercia) + silueta (muestra 20k)
# ----------------------------------------------------------------------------
ks = list(range(2, 11))
inertias, siluetas = [], []

rng = np.random.default_rng(RANDOM_STATE)
n_sample = min(20_000, X_scaled.shape[0])
sample_idx = rng.choice(X_scaled.shape[0], size=n_sample, replace=False)

print(f"\n{'k':>3}  {'inercia':>14}  {'silueta (n=20k)':>18}")
print("-" * 42)
for k in ks:
    mb = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                         batch_size=4096, n_init=5)
    labels = mb.fit_predict(X_scaled)
    inertias.append(mb.inertia_)
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    siluetas.append(sil)
    print(f"{k:>3}  {mb.inertia_:>14.2f}  {sil:>18.4f}")

k_best = ks[int(np.argmax(siluetas))]
print(f"\n>>> Mejor k por silueta: {k_best}  (silueta = {max(siluetas):.4f})")

# --- Gráficos codo + silueta ---
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(ks, inertias, "bo-")
axes[0].set_xlabel("$k$"); axes[0].set_ylabel("Inercia")
axes[0].set_title("Método del codo — Aprox. 1 (por IC)")
axes[0].grid(True)

axes[1].plot(ks, siluetas, "ro-")
axes[1].axvline(k_best, color="k", linestyle="--", alpha=0.4,
                label=f"Mejor k = {k_best}")
axes[1].set_xlabel("$k$"); axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silueta por número de clusters")
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

# ----------------------------------------------------------------------------
# 4. MODELO FINAL — MiniBatchKMeans con k elegido
# ----------------------------------------------------------------------------
mb_final = MiniBatchKMeans(n_clusters=k_best, random_state=RANDOM_STATE,
                           batch_size=4096, n_init=10)
labels_final = mb_final.fit_predict(X_scaled)
df["CLUSTER_IC"] = labels_final

# Distancia de cada IC a su centroide (para detectar atípicas)
dist = mb_final.transform(X_scaled)
df["DIST_CENTROIDE"] = dist[np.arange(len(dist)), labels_final].astype(np.float32)

# Centroides en escala original (interpretables)
centroides_orig = pd.DataFrame(
    scaler.inverse_transform(mb_final.cluster_centers_),
    columns=feature_names,
    index=[f"Cluster {i}" for i in range(k_best)]
)

print(f"\nMiniBatchKMeans final: k = {k_best}")
print(f"Inercia              : {mb_final.inertia_:.2f}")
print(f"Silueta (n=20k)      : "
      f"{silhouette_score(X_scaled[sample_idx], labels_final[sample_idx]):.4f}")
print("\nTamaño por cluster:")
print(df["CLUSTER_IC"].value_counts().sort_index().to_frame("n_ICs").to_string())

print("\n=== Centroides en escala ORIGINAL (numéricas) ===")
print(centroides_orig[num_cols].round(3).to_string())

# ----------------------------------------------------------------------------
# 5. PERFILADO — tabla numérica + composición
# ----------------------------------------------------------------------------
perfil = df.groupby("CLUSTER_IC").agg(
    n               = ("CLUSTER_IC", "size"),
    t_mediana_h     = ("TIEMPO_HORAS", "median"),
    t_media_h       = ("TIEMPO_HORAS", "mean"),
    pct_urgentes    = ("HCPRIINTER", lambda s: (s == 3).mean()),
    pct_nocturno    = ("ES_NOCTURNO", "mean"),
    pct_fin_semana  = ("ES_FIN_SEMANA", "mean"),
    hora_media      = ("HORA_DIA", "mean"),
).round(3)
print("\n=== PERFIL NUMÉRICO POR CLUSTER ===")
print(perfil.to_string())

for k in range(k_best):
    sub = df[df["CLUSTER_IC"] == k]
    print(f"\n{'─'*62}\n🟦 CLUSTER {k}  ·  n = {len(sub):,} ICs\n{'─'*62}")
    print("Top 10 AREA_GRUPO:")
    print(sub["AREA_GRUPO"].value_counts(normalize=True).head(10).round(3).to_string())
    print("\nDistribución por PRIORIDAD:")
    print(sub["PRIORIDAD"].value_counts(normalize=True).round(3).to_string())

# ----------------------------------------------------------------------------
# 6. VISUALIZACIÓN PCA 2D + heatmap de centroides
# ----------------------------------------------------------------------------
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled[sample_idx])
labels_sample = labels_final[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_sample,
                     s=4, cmap="tab10", alpha=0.5)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title(f"MiniBatchKMeans k={k_best} — PCA (muestra {n_sample:,})")
plt.colorbar(sc, ax=axes[0], label="Cluster")

sns.heatmap(centroides_orig[num_cols].T, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, cbar_kws={"label": "valor original"},
            ax=axes[1])
axes[1].set_title("Centroides en escala ORIGINAL")
axes[1].set_xlabel("Cluster")
plt.tight_layout(); plt.show()

# ----------------------------------------------------------------------------
# 7. RESUMEN FINAL
# ----------------------------------------------------------------------------
p99_dist = df["DIST_CENTROIDE"].quantile(0.99)
print("\n" + "="*62)
print("✅ RESUMEN FINAL — Aproximación 1 (por IC)")
print("="*62)
print(f"Algoritmo            : MiniBatchKMeans (k={k_best})")
print(f"Observaciones        : {len(df):,} interconsultas individuales")
print(f"Features totales     : {len(feature_names)}")
print(f"Silueta (n=20k)      : "
      f"{silhouette_score(X_scaled[sample_idx], labels_sample):.4f}")
print(f"Inercia              : {mb_final.inertia_:.2f}")
print(f"Dist. media centroide: {df['DIST_CENTROIDE'].mean():.3f}")
print(f"ICs outliers (p99)   : {(df['DIST_CENTROIDE'] > p99_dist).sum():,} "
      f"← candidatas a revisión")
print("="*62)

#— Separación núcleo/outliers y sub-clustering

Tras lo anterior MiniBatchKMeans global (k seleccionado por silueta, típicamente k≈8-9) identificamos que el modelo estaba contaminado por  áreas raras   que no encajaban en ningún cluster y forzaban   imprecision.

Esta celda separa las dos poblaciones usando el percentil 99 de `DIST_CENTROIDE` —el núcleo (99% del sistema, régimen estándar) y los outliers (1%, régimen atípico)— y aplica sub-clustering solo sobre los outliers para descubrir si son ruido homogéneo o micro-regímenes con estructura propia.

Incluye limpieza defensiva de NaN/inf  y comparativas visuales (tiempo, hora, área, prioridad) que justifican la separación antes de entrenar modelos independientes .

In [ ]:
# ----------------------------------------------------------------------------
# 1. CORTE POR PERCENTIL p99
# ----------------------------------------------------------------------------
p99 = df["DIST_CENTROIDE"].quantile(0.99)
df_core     = df[df["DIST_CENTROIDE"] <= p99].copy()
df_outliers = df[df["DIST_CENTROIDE"] >  p99].copy()

print(f"Umbral p99           : {p99:.3f}")
print(f"Media DIST_CENTROIDE : {df['DIST_CENTROIDE'].mean():.3f}")
print(f"Núcleo (inliers)     : {len(df_core):,}  "
      f"({100*len(df_core)/len(df):.2f}%)")
print(f"Outliers             : {len(df_outliers):,}  "
      f"({100*len(df_outliers)/len(df):.2f}%)")

# ----------------------------------------------------------------------------
# 2. COMPARATIVA NUMÉRICA
# ----------------------------------------------------------------------------
cols_cmp = ["TIEMPO_HORAS","LOG_TIEMPO","HCPRIINTER",
            "ES_NOCTURNO","ES_FIN_SEMANA","HORA_DIA"]
comp = pd.DataFrame({
    "Núcleo"  : df_core[cols_cmp].mean(),
    "Outliers": df_outliers[cols_cmp].mean(),
}).round(3)
comp["Δ (out - core)"] = (comp["Outliers"] - comp["Núcleo"]).round(3)
print("\n=== COMPARATIVA NÚCLEO vs OUTLIERS ===")
print(comp.to_string())

# ----------------------------------------------------------------------------
# 3. COMPOSICIÓN CATEGÓRICA
# ----------------------------------------------------------------------------
print("\n=== TOP AREA_GRUPO en OUTLIERS (proporción) ===")
print(df_outliers["AREA_GRUPO"].value_counts(normalize=True).head(10).round(3).to_string())

print("\n=== Áreas presentes en OUTLIERS vs TOTAL ===")
areas_tot = set(df["AREA_GRUPO"].unique())
areas_out = set(df_outliers["AREA_GRUPO"].unique())
print(f"  Áreas en outliers    : {sorted(areas_out)}")
print(f"  Áreas AUSENTES aquí  : {sorted(areas_tot - areas_out)}")
print(f"  → estas últimas coinciden con las áreas 'grandes' que "
      f"sí tienen su propio cluster")

print("\n=== PRIORIDAD (outliers vs núcleo, %) ===")
comp_pri = pd.DataFrame({
    "Núcleo"  : df_core["PRIORIDAD"].value_counts(normalize=True).round(3) * 100,
    "Outliers": df_outliers["PRIORIDAD"].value_counts(normalize=True).round(3) * 100,
}).round(1)
print(comp_pri.to_string())

# ----------------------------------------------------------------------------
# 4. VISUALIZACIÓN COMPARATIVA
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(np.log1p(df_core["TIEMPO_HORAS"]), bins=60, density=True,
             alpha=0.55, label=f"Núcleo (n={len(df_core):,})", color="steelblue")
axes[0].hist(np.log1p(df_outliers["TIEMPO_HORAS"]), bins=60, density=True,
             alpha=0.55, label=f"Outliers (n={len(df_outliers):,})", color="crimson")
axes[0].set_xlabel("log(1 + TIEMPO_HORAS)"); axes[0].set_ylabel("Densidad")
axes[0].set_title("Distribución del tiempo (log)"); axes[0].legend()

axes[1].hist(df_core["HORA_DIA"], bins=24, density=True,
             alpha=0.55, label="Núcleo", color="steelblue")
axes[1].hist(df_outliers["HORA_DIA"], bins=24, density=True,
             alpha=0.55, label="Outliers", color="crimson")
axes[1].set_xlabel("HORA_DIA"); axes[1].set_ylabel("Densidad")
axes[1].set_title("Distribución horaria (0–23 h)"); axes[1].legend()

top_areas = df_outliers["AREA_GRUPO"].value_counts(normalize=True).head(8).index
comp_areas = pd.DataFrame({
    "Núcleo"  : df_core["AREA_GRUPO"].value_counts(normalize=True).reindex(top_areas),
    "Outliers": df_outliers["AREA_GRUPO"].value_counts(normalize=True).reindex(top_areas),
}).fillna(0)
comp_areas.plot(kind="barh", ax=axes[2], color=["steelblue","crimson"],
                edgecolor="black", alpha=0.85)
axes[2].set_xlabel("Proporción dentro de la población")
axes[2].set_title("Composición por ÁREA (Top 8 de outliers)")
axes[2].invert_yaxis(); axes[2].legend()

plt.tight_layout(); plt.show()

# ----------------------------------------------------------------------------
# 5. SUB-CLUSTERING DE OUTLIERS  (parte corregida con limpieza defensiva)
# ----------------------------------------------------------------------------
num_cols = ["LOG_TIEMPO", "HORA_DIA", "ES_FIN_SEMANA",
            "ES_NOCTURNO", "HCPRIINTER"]
cat_cols = ["AREA_GRUPO"]

# ---- 5a. DIAGNÓSTICO: valores no finitos ----
print("\n=== DIAGNÓSTICO: valores no finitos en df_outliers ===")
chk = df_outliers[num_cols]
n_nan = chk.isna().sum()
n_inf = np.isinf(chk.select_dtypes(include=[np.number])).sum()
print("  NaN por columna :", n_nan.to_dict())
print("  Inf por columna :", n_inf.to_dict())
print(f"  Total NaN       : {n_nan.sum()}")
print(f"  Total Inf       : {n_inf.sum()}")

# ---- 5b. LIMPIEZA DEFENSIVA ----
n_before = len(df_outliers)
df_outliers = (
    df_outliers
    .replace([np.inf, -np.inf], np.nan)     # inf → NaN
    .dropna(subset=num_cols)                # quita filas con NaN en numéricas
    .reset_index(drop=True)
)
df_outliers["AREA_GRUPO"] = (
    df_outliers["AREA_GRUPO"].fillna("DESCONOCIDA").astype(str)
)
print(f"\n  Filas eliminadas: {n_before - len(df_outliers)}")
print(f"  Filas para sub-clustering: {len(df_outliers):,}")

# ---- 5c. CONSTRUCCIÓN DE X_out ----
X_out = pd.concat([
    df_outliers[num_cols].astype(np.float32).reset_index(drop=True),
    pd.get_dummies(df_outliers[cat_cols],
                   prefix=cat_cols, drop_first=False, dtype=np.float32),
], axis=1)

# Seguridad final: cualquier no-finito residual → 0
X_out = X_out.replace([np.inf, -np.inf], 0).fillna(0)

scaler_out   = StandardScaler()
X_out_scaled = scaler_out.fit_transform(X_out).astype(np.float32)

# Verificación dura
assert np.isfinite(X_out_scaled).all(), \
    "❌ Persisten valores no finitos en X_out_scaled"
print(f"  ✓ X_out_scaled: {X_out_scaled.shape}  |  "
      f"media≈{X_out_scaled.mean():.3f}  std≈{X_out_scaled.std():.3f}")

# ---- 5d. BARRIDO DE SUB-k ----
ks_out  = list(range(2, 7))
sils_out = []
for k in ks_out:
    mb = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                         batch_size=2048, n_init=10)
    lab = mb.fit_predict(X_out_scaled)
    sils_out.append(silhouette_score(X_out_scaled, lab))
    print(f"  sub-k={k}  silueta={sils_out[-1]:.4f}")

k_out = ks_out[int(np.argmax(sils_out))]
print(f"\n>>> Mejor sub-k: {k_out}  (silueta={max(sils_out):.4f})")

# ---- 5e. MODELO FINAL DE OUTLIERS ----
mb_out = MiniBatchKMeans(n_clusters=k_out, random_state=RANDOM_STATE,
                         batch_size=2048, n_init=10)
df_outliers["SUB_CLUSTER"] = mb_out.fit_predict(X_out_scaled)

perfil_out = df_outliers.groupby("SUB_CLUSTER").agg(
    n              = ("SUB_CLUSTER","size"),
    t_mediana_h    = ("TIEMPO_HORAS","median"),
    t_max_h        = ("TIEMPO_HORAS","max"),
    pct_urgentes   = ("HCPRIINTER", lambda s: (s == 3).mean()),
    pct_nocturno   = ("ES_NOCTURNO","mean"),
    pct_fin_semana = ("ES_FIN_SEMANA","mean"),
    hora_media     = ("HORA_DIA","mean"),
).round(3)
print("\n=== PERFIL DE LOS SUB-CLUSTERS DE OUTLIERS ===")
print(perfil_out.to_string())

print("\n=== ÁREA DOMINANTE por sub-cluster ===")
for sc in sorted(df_outliers["SUB_CLUSTER"].unique()):
    sub      = df_outliers[df_outliers["SUB_CLUSTER"] == sc]
    top_area = sub["AREA_GRUPO"].value_counts().idxmax()
    top_pct  = sub["AREA_GRUPO"].value_counts(normalize=True).max()
    print(f"  Sub-cluster {sc}: área={top_area:>8s} ({top_pct:.1%})   n={len(sub):,}")

# ----------------------------------------------------------------------------
# 6. RESUMEN
# ----------------------------------------------------------------------------
print("\n" + "="*62)
print("✅ RESUMEN — 2 poblaciones separadas")
print("="*62)
print(f"NÚCLEO          : {len(df_core):,}  (99% — régimen estándar)")
print(f"OUTLIERS        : {len(df_outliers):,}  (1% — régimen atípico)")
print(f"  Sub-clusters  : {k_out}")
print(f"  Candidatos >24h: {(df_outliers['TIEMPO_HORAS'] > 24).sum():,}")
print(f"  Urgentes (3)   : {(df_outliers['HCPRIINTER'] == 3).sum():,}")
print("="*62)

# — Doble MiniBatchKMeans

Separa las ICs en dos poblaciones por el p99 de `DIST_CENTROIDE`: **núcleo** (99%) y **outliers** (1%). Entrena un `MiniBatchKMeans` independiente para cada una con parámetros calibrados a su tamaño. Reporta centroides, perfiles, siluetas y el cruce con especialidades.

In [ ]:
# ----------------------------------------------------------------------------
# 0. DEFINICIÓN DE VARIABLES — Features numéricas y categóricas del modelo
# ----------------------------------------------------------------------------
NUM_COLS = ["LOG_TIEMPO", "HORA_DIA", "ES_FIN_SEMANA", "ES_NOCTURNO", "HCPRIINTER"]
CAT_COLS = ["AREA_GRUPO"]


# ============================================================================
# 1. SEPARACIÓN EN 2 POBLACIONES (p99 de DIST_CENTROIDE)
# ============================================================================
p99 = df["DIST_CENTROIDE"].quantile(0.99)
df_core     = df[df["DIST_CENTROIDE"] <= p99].copy()
df_outliers = df[df["DIST_CENTROIDE"] >  p99].copy()

print("=" * 70)
print("— Poblaciones núcleo vs outliers")
print("=" * 70)
print(f"Umbral p99            : {p99:.3f}")
print(f"Media DIST_CENTROIDE  : {df['DIST_CENTROIDE'].mean():.3f}")
print(f"NÚCLEO (inliers)      : {len(df_core):,}  "
      f"({100*len(df_core)/len(df):.2f}%)")
print(f"OUTLIERS              : {len(df_outliers):,}  "
      f"({100*len(df_outliers)/len(df):.2f}%)")


# ============================================================================
# 2. COMPARATIVA + VISUALIZACIÓN POBLACIONAL
# ============================================================================
cols_cmp = ["TIEMPO_HORAS", "LOG_TIEMPO", "HCPRIINTER",
            "ES_NOCTURNO", "ES_FIN_SEMANA", "HORA_DIA"]
comp = pd.DataFrame({
    "Núcleo"  : df_core[cols_cmp].mean(),
    "Outliers": df_outliers[cols_cmp].mean(),
}).round(3)
comp["Δ (out - core)"] = (comp["Outliers"] - comp["Núcleo"]).round(3)
print("\n=== COMPARATIVA NÚCLEO vs OUTLIERS ===")
print(comp.to_string())

print("\n=== Áreas en OUTLIERS ===")
areas_out = sorted(df_outliers["AREA_GRUPO"].unique())
print(f"  {areas_out}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(np.log1p(df_core["TIEMPO_HORAS"]), bins=60, density=True,
             alpha=0.55, label=f"Núcleo (n={len(df_core):,})", color="steelblue")
axes[0].hist(np.log1p(df_outliers["TIEMPO_HORAS"]), bins=60, density=True,
             alpha=0.55, label=f"Outliers (n={len(df_outliers):,})", color="crimson")
axes[0].set_xlabel("log(1 + TIEMPO_HORAS)"); axes[0].set_ylabel("Densidad")
axes[0].set_title("Tiempo (log)"); axes[0].legend()

axes[1].hist(df_core["HORA_DIA"], bins=24, density=True,
             alpha=0.55, label="Núcleo", color="steelblue")
axes[1].hist(df_outliers["HORA_DIA"], bins=24, density=True,
             alpha=0.55, label="Outliers", color="crimson")
axes[1].set_xlabel("HORA_DIA"); axes[1].set_ylabel("Densidad")
axes[1].set_title("Hora del día"); axes[1].legend()

top_areas = df_outliers["AREA_GRUPO"].value_counts(normalize=True).head(8).index
comp_areas = pd.DataFrame({
    "Núcleo"  : df_core["AREA_GRUPO"].value_counts(normalize=True).reindex(top_areas),
    "Outliers": df_outliers["AREA_GRUPO"].value_counts(normalize=True).reindex(top_areas),
}).fillna(0)
comp_areas.plot(kind="barh", ax=axes[2], color=["steelblue", "crimson"],
                edgecolor="black", alpha=0.85)
axes[2].set_xlabel("Proporción"); axes[2].set_title("Áreas (Top 8 de outliers)")
axes[2].invert_yaxis(); axes[2].legend()
plt.tight_layout(); plt.show()


# ============================================================================
# 3. UTILIDADES
# ============================================================================
def build_scaled(df_in, num_cols=NUM_COLS, cat_cols=CAT_COLS):
    """
    Limpieza defensiva + one-hot + StandardScaler.

    IMPORTANTE: preserva el índice original del DataFrame de entrada en la
    columna `_idx_orig`, para poder re-asignar etiquetas al df global sin
    perder la correspondencia tras el reset_index.
    """
    d = df_in.copy()

    # (a) Guardar índice original ANTES de cualquier reset
    d["_idx_orig"] = d.index

    # (b) Limpieza de numéricas
    d[num_cols] = (d[num_cols].astype(np.float32)
                                .replace([np.inf, -np.inf], np.nan))
    d = d.dropna(subset=num_cols).reset_index(drop=True)

    # (c) Categóricas
    d[cat_cols] = d[cat_cols].fillna("DESCONOCIDA").astype(str)

    # (d) Matriz X = numéricas + one-hot  (NO incluye _idx_orig)
    X = pd.concat([
        d[num_cols].reset_index(drop=True),
        pd.get_dummies(d[cat_cols], prefix=cat_cols,
                       drop_first=False, dtype=np.float32),
    ], axis=1)
    X = X.replace([np.inf, -np.inf], 0).fillna(0)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X).astype(np.float32)
    assert np.isfinite(Xs).all(), "❌ no finitos residuales"
    return d, Xs, scaler, X.columns.tolist()


def fit_minibatch(X_scaled, ks, batch_size, n_init, sample_idx, label=""):
    """Barrido codo+silueta, fit final con k*, devuelve modelo + labels."""
    print(f"\n{label} — barrido k = {ks}")
    print(f"  {'k':>3}  {'inercia':>14}  {'silueta':>10}")
    print("  " + "-" * 32)
    inercias, sils = [], []
    for k in ks:
        mb = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                             batch_size=batch_size, n_init=n_init)
        lab = mb.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled[sample_idx], lab[sample_idx])
        inercias.append(mb.inertia_); sils.append(sil)
        print(f"  {k:>3}  {mb.inertia_:>14.2f}  {sil:>10.4f}")

    k_best = ks[int(np.argmax(sils))]
    mb_final = MiniBatchKMeans(n_clusters=k_best, random_state=RANDOM_STATE,
                               batch_size=batch_size, n_init=n_init)
    labels = mb_final.fit_predict(X_scaled)
    sil_best = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    print(f"  → k*={k_best}  silueta={sil_best:.4f}  inercia={mb_final.inertia_:.2f}")
    return mb_final, labels, k_best, sil_best, inercias, sils


# ============================================================================
# 4. POBLACIÓN 1 — NÚCLEO  (batch grande, n_init=10, silueta sobre muestra)
# ============================================================================
print("\n" + "=" * 70)
print("🟦 POBLACIÓN 1 — NÚCLEO")
print("=" * 70)
df_core_c, X_core, scaler_core, feat_core = build_scaled(df_core)
print(f"  Filas : {len(df_core_c):,}  |  Features : {len(feat_core)}")

rng = np.random.default_rng(RANDOM_STATE)
n_s = min(20_000, len(X_core))
idx_core = rng.choice(len(X_core), n_s, replace=False)

mb_core, lab_core, k_core, sil_core, iner_c, sil_c = fit_minibatch(
    X_core, ks=[2, 3, 4, 5, 6, 7, 8, 9, 10],
    batch_size=4096, n_init=10,
    sample_idx=idx_core, label="Núcleo"
)
df_core_c["CLUSTER_CORE"] = lab_core

cent_core = pd.DataFrame(
    scaler_core.inverse_transform(mb_core.cluster_centers_),
    columns=feat_core, index=[f"C{i}" for i in range(k_core)]
)

perfil_core = df_core_c.groupby("CLUSTER_CORE").agg(
    n              = ("CLUSTER_CORE", "size"),
    t_mediana_h    = ("TIEMPO_HORAS", "median"),
    t_media_h      = ("TIEMPO_HORAS", "mean"),
    pct_urgentes   = ("HCPRIINTER", lambda s: (s == 3).mean()),
    pct_nocturno   = ("ES_NOCTURNO", "mean"),
    pct_fin_semana = ("ES_FIN_SEMANA", "mean"),
    hora_media     = ("HORA_DIA", "mean"),
).round(3)
print("\n=== PERFIL NÚCLEO ===")
print(perfil_core.to_string())
print("\n=== CENTROIDES NÚCLEO (numéricas originales) ===")
print(cent_core[NUM_COLS].round(3).to_string())


# ============================================================================
# 5. POBLACIÓN 2 — OUTLIERS  (batch chico, n_init=20, silueta exacta)
# ============================================================================
print("\n" + "=" * 70)
print("🟥 POBLACIÓN 2 — OUTLIERS")
print("=" * 70)
df_out_c, X_out, scaler_out, feat_out = build_scaled(df_outliers)
print(f"  Filas : {len(df_out_c):,}  |  Features : {len(feat_out)}")

mb_out, lab_out, k_out, sil_out, iner_o, sil_o = fit_minibatch(
    X_out, ks=[2, 3, 4, 5, 6],
    batch_size=256, n_init=20,
    sample_idx=np.arange(len(X_out)),   # silueta EXACTA (~4.5k filas)
    label="Outliers"
)
df_out_c["CLUSTER_OUT"] = lab_out

cent_out = pd.DataFrame(
    scaler_out.inverse_transform(mb_out.cluster_centers_),
    columns=feat_out, index=[f"O{i}" for i in range(k_out)]
)

perfil_out = df_out_c.groupby("CLUSTER_OUT").agg(
    n              = ("CLUSTER_OUT", "size"),
    t_mediana_h    = ("TIEMPO_HORAS", "median"),
    t_max_h        = ("TIEMPO_HORAS", "max"),
    pct_urgentes   = ("HCPRIINTER", lambda s: (s == 3).mean()),
    pct_nocturno   = ("ES_NOCTURNO", "mean"),
    pct_fin_semana = ("ES_FIN_SEMANA", "mean"),
    hora_media     = ("HORA_DIA", "mean"),
).round(3)
print("\n=== PERFIL OUTLIERS ===")
print(perfil_out.to_string())
print("\n=== CENTROIDES OUTLIERS (numéricas originales) ===")
print(cent_out[NUM_COLS].round(3).to_string())

print("\n=== ÁREA DOMINANTE por sub-cluster ===")
for sc in sorted(df_out_c["CLUSTER_OUT"].unique()):
    sub = df_out_c[df_out_c["CLUSTER_OUT"] == sc]
    top = sub["AREA_GRUPO"].value_counts()
    print(f"  O{sc}: {top.index[0]:>8s} ({top.iloc[0]/len(sub):.1%})  n={len(sub):,}")


# ============================================================================
# 6. VISUALIZACIÓN FINAL: heatmaps de centroides lado a lado
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cent_core[NUM_COLS].T, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=axes[0],
            cbar_kws={"label": "valor original"})
axes[0].set_title(f"Centroides NÚCLEO (k={k_core}, sil={sil_core:.3f})")
axes[0].set_xlabel("Cluster")

sns.heatmap(cent_out[NUM_COLS].T, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=axes[1],
            cbar_kws={"label": "valor original"})
axes[1].set_title(f"Centroides OUTLIERS (k={k_out}, sil={sil_out:.3f})")
axes[1].set_xlabel("Sub-cluster")
plt.tight_layout(); plt.show()


# ============================================================================
# 7. UNIÓN CORREGIDA — usar _idx_orig (índice original de df) en lugar
#    del índice post-reset_index de build_scaled
# ============================================================================
df["CLUSTER_FINAL"] = -1
df["MODELO_ORIGEN"] = "sin_asignar"

# --- Núcleo ---
idx_core_global = df_core_c["_idx_orig"].values
df.loc[idx_core_global, "CLUSTER_FINAL"] = df_core_c["CLUSTER_CORE"].values
df.loc[idx_core_global, "MODELO_ORIGEN"] = "nucleo"

# --- Outliers ---
idx_out_global = df_out_c["_idx_orig"].values
df.loc[idx_out_global, "CLUSTER_FINAL"] = df_out_c["CLUSTER_OUT"].values
df.loc[idx_out_global, "MODELO_ORIGEN"] = "outlier"

df["ID_GLOBAL"] = df["MODELO_ORIGEN"] + "_" + df["CLUSTER_FINAL"].astype(str)


# ============================================================================
# 7b. VERIFICACIÓN DURA DE LA ASIGNACIÓN
# ============================================================================
n_sin_asignar   = (df["MODELO_ORIGEN"] == "sin_asignar").sum()
n_nucleo        = (df["MODELO_ORIGEN"] == "nucleo").sum()
n_outlier       = (df["MODELO_ORIGEN"] == "outlier").sum()
n_perdidas      = len(df) - n_sin_asignar - n_nucleo - n_outlier

assert n_perdidas == 0, "❌ Error de conteo en la asignación"
assert n_nucleo  == len(df_core_c),  \
    f"❌ Núcleo asignado ({n_nucleo:,}) ≠ filas del modelo ({len(df_core_c):,})"
assert n_outlier == len(df_out_c),   \
    f"❌ Outliers asignados ({n_outlier:,}) ≠ filas del modelo ({len(df_out_c):,})"

# Chequeo semántico: las filas marcadas 'nucleo' deben tener DIST_CENTROIDE <= p99
assert (df.loc[df["MODELO_ORIGEN"] == "nucleo", "DIST_CENTROIDE"] <= p99).all(), \
    "❌ Hay filas 'nucleo' con distancia > p99 (asignación cruzada)"
assert (df.loc[df["MODELO_ORIGEN"] == "outlier", "DIST_CENTROIDE"] > p99).all(), \
    "❌ Hay filas 'outlier' con distancia <= p99 (asignación cruzada)"

print("✓ Verificación de asignación OK:")
print(f"    núcleo   : {n_nucleo:,}")
print(f"    outlier  : {n_outlier:,}")
print(f"    sin asig.: {n_sin_asignar:,}  (esperado: filas descartadas por limpieza defensiva)")


# ============================================================================
# 8. RESUMEN FINAL
# ============================================================================
print("\n" + "=" * 70)
print("✅ RESUMEN GLOBAL — Modelos separados")
print("=" * 70)
print(f"NÚCLEO     → k={k_core:>2}  silueta={sil_core:.4f}  n={len(df_core_c):,}")
print(f"OUTLIERS   → k={k_out:>2}  silueta={sil_out:.4f}  n={len(df_out_c):,}")
print(f"Sin asignar: {len(df) - len(df_core_c) - len(df_out_c):,} "
      f"(por limpieza defensiva)")
print("\nDistribución MODELO_ORIGEN:")
print(df["MODELO_ORIGEN"].value_counts().to_string())
print("\nDistribución ID_GLOBAL (top 15):")
print(df["ID_GLOBAL"].value_counts().head(15).to_string())
print("=" * 70)

pd.crosstab(
    df.loc[df["MODELO_ORIGEN"] == "nucleo", "ESPECIALIDAD"],
    df.loc[df["MODELO_ORIGEN"] == "nucleo", "CLUSTER_FINAL"],
    normalize="index"
).round(2).head(25)

 # Expresión gráfica de los clusters

Los heatmaps de centroides ya muestran qué caracteriza a cada cluster, pero
no cómo se distribuyen las ICs en el espacio de features. Esta celda
complementa el análisis con visualizaciones.

## Visualizaciones incluidas

1. **PCA 2D — Núcleo (`k` elegido por silueta)** — proyección de las ICs del
   núcleo (≈99% del total) sobre las 2 primeras componentes principales,
   coloreadas por cluster. Muestra la **separación global** de los regímenes
   operativos.

2. **PCA 2D — Outliers (`k` elegido por silueta)** — proyección de las ICs
   atípicas (≈1% del total) sobre las 2 primeras componentes principales,
   coloreadas por sub-cluster. Al ser pocos puntos, se distinguen los
   micro-regímenes con claridad.

3. **Silhouette diagram — Núcleo  ** — diagrama de silueta
   por cluster con la media global. Permite ver **qué clusters están bien
   separados** (silueta cercana a +1) y cuáles son ambiguos (silueta < 0).

4. **Silhouette diagram — Outliers** — mismo diagrama aplicado al sub-modelo
   de outliers, con silueta exacta (sin muestreo) porque el subconjunto es
   pequeño.

In [ ]:
# ============================================================================
# 1. PCA 2D — NÚCLEO  (k=k_core)
# ============================================================================
# Muestra estratificada para no saturar el scatter (mantiene proporciones)
n_plot_core = 30_000
rng = np.random.default_rng(RANDOM_STATE)
idx_plot_core = rng.choice(len(X_core), size=min(n_plot_core, len(X_core)),
                           replace=False)

pca_core = PCA(n_components=2, random_state=RANDOM_STATE)
X_core_pca = pca_core.fit_transform(X_core[idx_plot_core])
lab_core_plot = lab_core[idx_plot_core]

fig, ax = plt.subplots(figsize=(11, 7))
palette = sns.color_palette("tab10", n_colors=k_core)

for k in range(k_core):
    mask = lab_core_plot == k
    ax.scatter(X_core_pca[mask, 0], X_core_pca[mask, 1],
               s=5, alpha=0.45, color=palette[k],
               label=f"C{k} (n={(lab_core == k).sum():,})")

# Centroides en el espacio PCA
cent_pca = pca_core.transform(mb_core.cluster_centers_)
ax.scatter(cent_pca[:, 0], cent_pca[:, 1],
           marker="X", s=260, c="black", edgecolor="white",
           linewidth=2, zorder=10, label="Centroides")

ax.set_xlabel(f"PC1 ({pca_core.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_core.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"PCA 2D — NÚCLEO  (k={k_core}, silueta={sil_core:.3f}, "
             f"muestra {len(idx_plot_core):,} ICs)")
ax.legend(loc="best", fontsize=9, markerscale=2.5, framealpha=0.9)
plt.tight_layout(); plt.show()


# ============================================================================
# 2. PCA 2D — OUTLIERS  (k=k_out, todos los puntos)
# ============================================================================
pca_out = PCA(n_components=2, random_state=RANDOM_STATE)
X_out_pca = pca_out.fit_transform(X_out_scaled)

fig, ax = plt.subplots(figsize=(11, 7))
palette_out = sns.color_palette("Set1", n_colors=k_out)

for k in range(k_out):
    mask = lab_out == k
    # Área dominante de este sub-cluster
    top_area = df_out_c.loc[mask, "AREA_GRUPO"].value_counts().idxmax()
    ax.scatter(X_out_pca[mask, 0], X_out_pca[mask, 1],
               s=14, alpha=0.7, color=palette_out[k],
               label=f"O{k} · {top_area} (n={mask.sum():,})")

# Centroides outliers en el espacio PCA
cent_out_pca = pca_out.transform(mb_out.cluster_centers_)
ax.scatter(cent_out_pca[:, 0], cent_out_pca[:, 1],
           marker="X", s=300, c="black", edgecolor="white",
           linewidth=2, zorder=10, label="Centroides")

ax.set_xlabel(f"PC1 ({pca_out.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca_out.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"PCA 2D — OUTLIERS  (k={k_out}, silueta={sil_out:.3f}, "
             f"n={len(X_out_scaled):,})")
ax.legend(loc="best", fontsize=9, markerscale=2, framealpha=0.9)
plt.tight_layout(); plt.show()


# ============================================================================
# 3. SILHOUETTE DIAGRAM — NÚCLEO  (Fig. 8-9 del libro)
# ============================================================================
# Usamos la MISMA muestra usada para calcular la silueta
sil_vals_core  = silhouette_samples(X_core[idx_core], lab_core[idx_core])
sil_mean_core  = silhouette_score(X_core[idx_core], lab_core[idx_core])
y_pred_core    = lab_core[idx_core]

plt.figure(figsize=(11, 8))
padding = len(X_core[idx_core]) // 40
pos = padding
ticks = []

for i in range(k_core):
    coeffs = sil_vals_core[y_pred_core == i]
    coeffs.sort()
    color = palette[i]
    plt.fill_betweenx(np.arange(pos, pos + len(coeffs)),
                      0, coeffs,
                      facecolor=color, edgecolor=color, alpha=0.75)
    ticks.append(pos + len(coeffs) // 2)
    pos += len(coeffs) + padding

plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
plt.gca().yaxis.set_major_formatter(FixedFormatter(
    [f"C{i} (n={(y_pred_core == i).sum():,})" for i in range(k_core)]
))
plt.axvline(x=sil_mean_core, color="red", linestyle="--",
            label=f"Media = {sil_mean_core:.3f}")
plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente de silueta")
plt.ylabel("Cluster")
plt.title(f"Silhouette diagram — NÚCLEO  (k={k_core}, "
          f"muestra n={len(idx_core):,})")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


# ============================================================================
# 4. SILHOUETTE DIAGRAM — OUTLIERS  (silueta exacta, sin muestreo)
# ============================================================================
sil_vals_out = silhouette_samples(X_out_scaled, lab_out)
sil_mean_out = silhouette_score(X_out_scaled, lab_out)

plt.figure(figsize=(11, 7))
padding = max(1, len(X_out_scaled) // 40)
pos = padding
ticks = []

for i in range(k_out):
    coeffs = sil_vals_out[lab_out == i]
    coeffs.sort()
    color = palette_out[i]
    plt.fill_betweenx(np.arange(pos, pos + len(coeffs)),
                      0, coeffs,
                      facecolor=color, edgecolor=color, alpha=0.8)
    top_area = df_out_c.loc[lab_out == i, "AREA_GRUPO"].value_counts().idxmax()
    ticks.append((pos + len(coeffs) // 2, f"O{i} · {top_area}"))
    pos += len(coeffs) + padding

plt.gca().yaxis.set_major_locator(FixedLocator([t[0] for t in ticks]))
plt.gca().yaxis.set_major_formatter(FixedFormatter([t[1] for t in ticks]))
plt.axvline(x=sil_mean_out, color="red", linestyle="--",
            label=f"Media = {sil_mean_out:.3f}")
plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente de silueta")
plt.ylabel("Sub-cluster")
plt.title(f"Silhouette diagram — OUTLIERS  (k={k_out}, "
          f"n={len(X_out_scaled):,}, silueta exacta)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


# ============================================================================
# 5. RESUMEN NUMÉRICO DE SILUETAS POR CLUSTER
# ============================================================================
print("=" * 66)
print("📊 SILUETA MEDIA POR CLUSTER")
print("=" * 66)

print("\n🟦 NÚCLEO (k={}):".format(k_core))
print(f"  {'Cluster':>8}  {'n':>10}  {'silueta media':>14}  {'% < 0':>8}")
for i in range(k_core):
    m = y_pred_core == i
    s = sil_vals_core[m]
    pct_neg = 100 * (s < 0).mean()
    print(f"  {('C'+str(i)):>8}  {m.sum():>10,}  {s.mean():>14.3f}  {pct_neg:>7.1f}%")

print(f"\n🟥 OUTLIERS (k={k_out}):")
print(f"  {'Cluster':>8}  {'n':>10}  {'silueta media':>14}  {'% < 0':>8}")
for i in range(k_out):
    m = lab_out == i
    s = sil_vals_out[m]
    pct_neg = 100 * (s < 0).mean()
    top_area = df_out_c.loc[m, "AREA_GRUPO"].value_counts().idxmax()
    print(f"  {('O'+str(i)):>8}  {m.sum():>10,}  {s.mean():>14.3f}  {pct_neg:>7.1f}%"
          f"   ← {top_area}")
print("=" * 66)

# ============================================================================
# P.D. Para Guardar los Resultados
# ============================================================================
# ============================================================================
# EXPORT FINAL — empaqueta todo y descarga ZIP
# ============================================================================
import zipfile, shutil
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

# --- 1. Auto-captura de DataFrames que quedaron en el scope global ---
_skip = {"pd", "np", "plt", "sns", "sklearn", "matplotlib", "OUT_DIR",
         "FIG_DIR", "TBL_DIR", "_TABLES_REGISTRY"}
for _name, _obj in list(globals().items()):
    if _name.startswith("_") or _name in _skip:
        continue
    if isinstance(_obj, pd.DataFrame) and len(_obj) > 0 \
            and _name not in _TABLES_REGISTRY:
        try:
            _TABLES_REGISTRY[f"auto_{_slug(_name)}"] = _obj.copy()
        except Exception:
            pass

# --- 2. Exportar todas las tablas registradas ---
def _export_table(name, obj):
    base = _slug(name)
    try:
        if isinstance(obj, pd.DataFrame):
            obj.to_csv(TBL_DIR / f"{base}.csv", index=True)
            try:
                obj.to_excel(TBL_DIR / f"{base}.xlsx", index=True)
            except Exception:
                pass  # openpyxl puede faltar, no es crítico
        elif isinstance(obj, pd.Series):
            s = obj.to_frame(name="valor")
            s.to_csv(TBL_DIR / f"{base}.csv", index=True)
            try:
                s.to_excel(TBL_DIR / f"{base}.xlsx", index=True)
            except Exception:
                pass
        else:
            with open(TBL_DIR / f"{base}.txt", "w", encoding="utf-8") as f:
                f.write(str(obj))
    except Exception as e:
        print(f"⚠ tabla '{name}': {e}")

n_tablas = 0
for name, obj in _TABLES_REGISTRY.items():
    _export_table(name, obj)
    n_tablas += 1

# --- 3. Guardar figuras que sigan abiertas (por si acaso) ---
for num in plt.get_fignums():
    fig = plt.figure(num)
    _FIG_COUNTER["n"] += 1
    n = _FIG_COUNTER["n"]
    base = f"fig{n:03d}_abierta_al_final"
    for ext in ("svg", "pdf", "png"):
        try:
            fig.savefig(FIG_DIR / f"{base}.{ext}", format=ext,
                        bbox_inches="tight", dpi=150, facecolor="white")
        except Exception:
            pass

# --- 4. README con metadatos ---
n_svg = len(list(FIG_DIR.glob("*.svg")))
n_pdf = len(list(FIG_DIR.glob("*.pdf")))
n_csv = len(list(TBL_DIR.glob("*.csv")))
with open(OUT_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write("Taller K-Means — export de resultados\n")
    f.write(f"Generado: {datetime.now().isoformat(timespec='seconds')}\n\n")
    f.write(f"Figuras : {n_svg} SVG · {n_pdf} PDF · "
            f"{len(list(FIG_DIR.glob('*.png')))} PNG\n")
    f.write(f"Tablas  : {n_csv} CSV · "
            f"{len(list(TBL_DIR.glob('*.xlsx')))} XLSX\n\n")
    f.write("Estructura:\n")
    f.write("  figures/  → SVG (vector editable), PDF (vector), PNG (vista rápida)\n")
    f.write("  tables/   → CSV + XLSX\n")

# --- 5. ZIP ---
zip_path = Path("taller_kmeans_export.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUT_DIR.parent))

print("=" * 60)
print("✅ EXPORT LISTO")
print("=" * 60)
print(f"  Figuras : {n_svg} SVG + {n_pdf} PDF")
print(f"  Tablas  : {n_csv} CSV + {n_tablas} registradas")
print(f"  ZIP     : {zip_path.resolve()}  "
      f"({zip_path.stat().st_size/1024/1024:.2f} MB)")

# --- 6. Descarga (Colab) o link (local) ---
try:
    from google.colab import files as _colab_files
    _colab_files.download(str(zip_path))
except Exception:
    display(FileLink(str(zip_path)))